# SEACO-6161 · 05 Savings report · `dbw-opas-merone-pipeline-prod`

**Read-only.** Answers *"how much storage did the cleanup clear on `dbstorageld3xv3o6mfr6o`, and how much money does that
save?"* It compares the state **before** the cleanup with the state **after** it, for both kinds of storage:

- **Table storage**: Delta and other tables registered in the metastore whose data is on the root account
  (VACUUM, `drop_tables`, `drop_databases`, `03_weekly_vacuum_job`).
- **File storage**: everything else under `dbfs:/` (orphan directories, cluster logs, `/FileStore`, `/tmp`, …)
  (`delete_orphan_dirs`, `prune_cluster_logs`, and any folders removed by hand).

Every size is turned into **$/month and $/year** at the account's effective rate
($$2,189 / 188.41 TiB from Azure, May 2026).

| Section | Source | Question |
|---|---|---|
| 1 | all audit tables | Which runs count as *before* and *after* the cleanup? |
| 2 | `cleanup_log` | What did each cleanup step remove (bytes, files, $), measured when it ran? |
| 3 | `table_inventory` before vs after | How did table storage change, per database? Which tables are gone? |
| 4 | `dbfs_scan` before vs after | How did file storage change, per `dbfs:/` directory? |
| 5 | `baseline` + Azure | Account totals before and after, and the real Azure figure if you enter it |
| 6 | all of the above | Summary, printed as Markdown for the ticket |

**How to get a good *after* picture:** after the last cleanup step, run `01_investigation` and `01a_dbfs_scan`
again (resume `01a` to 100% coverage), then run this notebook. Leave the run widgets empty: *before* is the
latest run started before the first real cleanup action, *after* is the latest run started after the last one.
Azure storage metrics lag by about a day, and its cost by a billing cycle; enter the new Azure size in
`azure_now_tib` for the real figure.

In [0]:
%run ./00_config_and_helpers

In [0]:
dbutils.widgets.text("before_inventory_run", "", "1. Before: table_inventory run_id (empty = auto)")
dbutils.widgets.text("after_inventory_run", "", "2. After: table_inventory run_id (empty = auto)")
dbutils.widgets.text("before_scan_run", "", "3. Before: dbfs_scan run_id (empty = auto)")
dbutils.widgets.text("after_scan_run", "", "4. After: dbfs_scan run_id (empty = auto)")
dbutils.widgets.text("azure_now_tib", "", "5. Account size in Azure now, TiB (optional)")

W_INV_B = dbutils.widgets.get("before_inventory_run").strip()
W_INV_A = dbutils.widgets.get("after_inventory_run").strip()
W_SCAN_B = dbutils.widgets.get("before_scan_run").strip()
W_SCAN_A = dbutils.widgets.get("after_scan_run").strip()
AZURE_NOW_TIB = float(dbutils.widgets.get("azure_now_tib")) if dbutils.widgets.get("azure_now_tib").strip() else None

## 1 · Before and after runs

In [0]:
RATE = AZURE_MONTHLY_COST_USD / (AZURE_REPORTED_TIB * TIB)   # $ per byte-month
FILE_STEPS = ("delete_orphan_dirs", "prune_cluster_logs")    # everything else in cleanup_log is table storage


def usd(b):
    return 0.0 if b is None else float(b) * RATE


def n0(v):
    return v or 0


def table_or_none(name):
    try:
        return spark.table(audit_table(name))
    except Exception:
        print(f"  ! {AUDIT_DB}.{name} not found")
        return None


spark.udf.register("usd", lambda b: None if b is None else round(float(b) * RATE, 2), "double")

# ---- Cleanup actions that really happened --------------------------------------------------------------
log = table_or_none("cleanup_log")
if log is None:
    raise RuntimeError("cleanup_log is empty: nothing has been cleaned up yet")
for c in ("files_before", "files_after"):  # added after the first cleanup runs; older rows have none
    if c not in log.columns:
        log = log.withColumn(c, F.lit(None).cast("long"))
log = log.withColumn("storage", F.when(F.col("step").isin(*FILE_STEPS), "file").otherwise("table"))
done = log.filter("NOT dry_run AND status = 'ok'").cache()
done.createOrReplaceTempView("done")
FIRST_TS, LAST_TS = done.agg(F.min("action_ts"), F.max("action_ts")).first()


# ---- Before / after runs ---------------------------------------------------------------------------------
def runs(name):
    df = table_or_none(name)
    if df is None:
        return []
    return [r.asDict() for r in df.groupBy("run_id").agg(F.min("run_ts").alias("run_ts")).orderBy("run_ts").collect()]


def pick(name, wanted, side):
    rs = runs(name)
    if wanted:
        m = [r for r in rs if r["run_id"] == wanted]
        if not m:
            raise ValueError(f"{AUDIT_DB}.{name}: run_id {wanted} not found")
        return m[0]
    if not rs:
        return None
    if FIRST_TS is None:  # no real cleanup yet: compare first and latest run
        return rs[0] if side == "before" else rs[-1]
    if side == "before":
        c = [r for r in rs if r["run_ts"] < FIRST_TS]
    else:
        c = [r for r in rs if r["run_ts"] > LAST_TS]
    return c[-1] if c else None


INV_B, INV_A = pick("table_inventory", W_INV_B, "before"), pick("table_inventory", W_INV_A, "after")
SCAN_B, SCAN_A = pick("dbfs_scan_plan", W_SCAN_B, "before"), pick("dbfs_scan_plan", W_SCAN_A, "after")

inv_all = table_or_none("table_inventory")
scan_all = table_or_none("dbfs_scan")
plan_all = table_or_none("dbfs_scan_plan")


def inv_stats(run):
    if run is None:
        return None
    df = inv_all.filter(F.col("run_id") == run["run_id"])
    return df.agg(F.count("*").alias("tables"), F.countDistinct("database").alias("dbs")).first().asDict()


def scan_coverage(run):
    if run is None or scan_all is None:
        return None
    planned = plan_all.filter((F.col("run_id") == run["run_id"]) & (F.col("kind") == "dir")
                              & (F.col("storage_class") == "root")).count()
    sized = scan_all.filter((F.col("run_id") == run["run_id"]) & (F.col("kind") == "dir")
                            & (F.col("storage_class") == "root")).select("path").distinct().count()
    return sized / max(planned, 1)


INV_STATS = dict(before=inv_stats(INV_B), after=inv_stats(INV_A))
SCAN_COV = dict(before=scan_coverage(SCAN_B), after=scan_coverage(SCAN_A))

In [0]:
def show_run(label, run, extra=""):
    if run is None:
        print(f"{label:<28}: none found {extra}")
    else:
        print(f"{label:<28}: {run['run_id']} @ {run['run_ts']:%Y-%m-%d %H:%M} UTC {extra}")


print(f"Effective rate              : ${RATE * GIB:.5f} per GiB-month (${AZURE_MONTHLY_COST_USD:,.2f} / {AZURE_REPORTED_TIB} TiB)")
if FIRST_TS is None:
    print("Real cleanup actions        : none yet (only dry runs). Comparing the first and the latest runs.")
else:
    print(f"Real cleanup actions        : {done.count():,} between {FIRST_TS:%Y-%m-%d %H:%M} and {LAST_TS:%Y-%m-%d %H:%M} UTC")
for side, run in (("before", INV_B), ("after", INV_A)):
    s = INV_STATS[side]
    show_run(f"table_inventory {side}", run, f"({s['tables']:,} tables in {s['dbs']} databases)" if s else "")
for side, run in (("before", SCAN_B), ("after", SCAN_A)):
    cov = SCAN_COV[side]
    show_run(f"dbfs_scan {side}", run, f"(coverage {cov:.1%})" if cov is not None else "")

if INV_A is None or SCAN_A is None:
    print("\n⚠️  No run after the last cleanup action. Run 01_investigation and 01a_dbfs_scan again; until then "
          "sections 3 and 4 are empty and only section 2 (measured during the cleanup) counts.")
if INV_B is None or SCAN_B is None:
    print("\n⚠️  No run before the first cleanup action. Sections 3 and 4 need one; section 2 still counts.")
if INV_STATS["before"] and INV_STATS["after"] and INV_STATS["before"]["dbs"] != INV_STATS["after"]["dbs"]:
    print("\n⚠️  The before and after inventories cover a different number of databases. If one of them was "
          "limited with the `databases` widget, section 3 compares unlike things. Dropped databases also lower "
          "the after count, which is expected.")
for side in ("before", "after"):
    if SCAN_COV[side] is not None and SCAN_COV[side] < 0.999:
        print(f"\n⚠️  The {side} dbfs_scan covered only {SCAN_COV[side]:.0%}: directories it did not size are listed "
              f"as 'not sized' in section 4 and left out of the totals. Resume it to 100%.")

## 2 · What the cleanup removed (`cleanup_log`)

Measured by each step at the moment it ran: size and file count of the target just before and just after.
This is the most reliable number, because it does not depend on a full scan. Only real actions count
(`dry_run = false`, `status = 'ok'`).

In [0]:
STEP_SUMMARY = spark.sql("""
    SELECT storage, step,
           count(*)                                                         AS actions,
           round(sum(bytes_before - coalesce(bytes_after, bytes_before)) / power(1024, 4), 3) AS freed_tib,
           usd(sum(bytes_before - coalesce(bytes_after, bytes_before)))     AS saved_usd_per_month,
           usd(sum(bytes_before - coalesce(bytes_after, bytes_before))) * 12 AS saved_usd_per_year,
           sum(files_before)                                                AS files_before,
           sum(files_after)                                                 AS files_after,
           sum(files_before - coalesce(files_after, files_before))          AS files_removed,
           min(action_ts) AS first_action, max(action_ts) AS last_action
    FROM done
    GROUP BY storage, step
    ORDER BY storage DESC, freed_tib DESC
""").cache()
display(STEP_SUMMARY)

In [0]:
tot = {r["storage"]: r.asDict() for r in spark.sql("""
    SELECT storage, sum(bytes_before - coalesce(bytes_after, bytes_before)) AS freed,
           sum(files_before - coalesce(files_after, files_before)) AS files, count(*) AS actions
    FROM done GROUP BY storage""").collect()}
ACT = dict(table=tot.get("table", {}), file=tot.get("file", {}))
ACT_FREED = n0(ACT["table"].get("freed")) + n0(ACT["file"].get("freed"))
ACT_FILES = n0(ACT["table"].get("files")) + n0(ACT["file"].get("files"))
print(f"{'Measured during the cleanup':<34} {'freed':>12} {'files removed':>15} {'$/month':>10} {'$/year':>11}")
for k, label in (("table", "Table storage"), ("file", "File storage (dbfs:/ folders)")):
    b, f = n0(ACT[k].get("freed")), n0(ACT[k].get("files"))
    print(f"{label:<34} {human(b):>12} {f:>15,} {usd(b):>10,.0f} {usd(b) * 12:>11,.0f}")
print(f"{'Total':<34} {human(ACT_FREED):>12} {ACT_FILES:>15,} {usd(ACT_FREED):>10,.0f} {usd(ACT_FREED) * 12:>11,.0f}")
print("\nFile counts are only recorded for actions run after file counting was added to 02_cleanup / 03.")

In [0]:
%sql
-- The 100 actions that freed most
SELECT storage, step, target, action_ts,
       round(bytes_before / power(1024, 3), 2)                                  AS before_gib,
       round(coalesce(bytes_after, bytes_before) / power(1024, 3), 2)           AS after_gib,
       round((bytes_before - coalesce(bytes_after, bytes_before)) / power(1024, 3), 2) AS freed_gib,
       usd(bytes_before - coalesce(bytes_after, bytes_before))                  AS saved_usd_per_month,
       files_before, files_after, files_before - coalesce(files_after, files_before) AS files_removed,
       run_user
FROM done
ORDER BY bytes_before - coalesce(bytes_after, bytes_before) DESC
LIMIT 100

## 3 · Table storage, before vs after (`table_inventory`)

Root-account tables in the *before* inventory compared with the *after* inventory. A table missing from the
after inventory counts as dropped (size after = 0). `measure` says whether the on-disk size or the live size was
compared; see the note under the totals.

In [0]:
def inv_side(run, side):
    cols = ["database", "full_name", "disk_bytes", "disk_files", "live_bytes", "num_files"]
    if run is None:
        return spark.createDataFrame([], f"database string, full_name string, disk_bytes_{side} long, "
                                         f"disk_files_{side} long, live_bytes_{side} long, num_files_{side} long, "
                                         f"present_{side} boolean")
    df = inv_all.filter((F.col("run_id") == run["run_id"]) & (F.col("storage_class") == "root")) \
        .dropDuplicates(["full_name"]).select(*cols)
    return df.select("database", "full_name", *[F.col(c).alias(f"{c}_{side}") for c in cols[2:]],
                     F.lit(True).alias(f"present_{side}"))


tb, ta = inv_side(INV_B, "b"), inv_side(INV_A, "a")
TBL = tb.join(ta, ["database", "full_name"], "full_outer").withColumn(
    "change", F.when(F.col("present_a").isNull(), "dropped").when(F.col("present_b").isNull(), "new")
               .otherwise("kept"))
# On-disk size is the measure. When on-disk was not sized on either side, the live size is used instead,
# and the row says so: the live size of a kept table hardly changes with VACUUM, so that understates savings.
TBL = TBL.withColumn("measure", F.when(F.col("disk_bytes_b").isNotNull() & (F.col("disk_bytes_a").isNotNull()
                                                                          | (F.col("change") == "dropped")), "disk")
                                 .otherwise("live")) \
    .withColumn("bytes_b", F.when(F.col("measure") == "disk", F.col("disk_bytes_b")).otherwise(F.col("live_bytes_b"))) \
    .withColumn("bytes_a", F.when(F.col("change") == "dropped", F.lit(0))
                .when(F.col("measure") == "disk", F.col("disk_bytes_a")).otherwise(F.col("live_bytes_a"))) \
    .withColumn("files_b", F.coalesce("disk_files_b", "num_files_b")) \
    .withColumn("files_a", F.when(F.col("change") == "dropped", F.lit(0)).otherwise(F.coalesce("disk_files_a", "num_files_a")))
TBL.cache().createOrReplaceTempView("tbl_cmp")
DB_CMP = spark.sql("""
    SELECT database,
           count_if(change != 'new')     AS tables_before,
           count_if(change != 'dropped') AS tables_after,
           count_if(change = 'dropped')  AS tables_dropped,
           round(sum(coalesce(bytes_b, 0)) / power(1024, 4), 3) AS before_tib,
           round(sum(coalesce(bytes_a, 0)) / power(1024, 4), 3) AS after_tib,
           round(sum(coalesce(bytes_b, 0) - coalesce(bytes_a, 0)) / power(1024, 4), 3) AS saved_tib,
           usd(sum(coalesce(bytes_b, 0) - coalesce(bytes_a, 0))) AS saved_usd_per_month,
           sum(files_b) AS files_before, sum(files_a) AS files_after,
           sum(coalesce(files_b, 0) - coalesce(files_a, 0)) AS files_removed,
           count_if(measure = 'live') AS tables_measured_by_live_size
    FROM tbl_cmp GROUP BY database
    ORDER BY saved_tib DESC
""").cache()
display(DB_CMP)

In [0]:
%sql
-- Tables in the before inventory that are gone from the after inventory
SELECT database, full_name, measure,
       round(bytes_b / power(1024, 3), 2) AS size_gib, usd(bytes_b) AS saved_usd_per_month, files_b AS files
FROM tbl_cmp
WHERE change = 'dropped'
ORDER BY bytes_b DESC NULLS LAST

In [0]:
T = spark.sql("""
    SELECT sum(coalesce(bytes_b, 0)) AS b, sum(coalesce(bytes_a, 0)) AS a,
           sum(coalesce(files_b, 0)) AS fb, sum(coalesce(files_a, 0)) AS fa,
           count_if(change = 'dropped') AS dropped, count_if(measure = 'live') AS live_measured, count(*) AS n
    FROM tbl_cmp""").first().asDict()
TBL_SAVED = n0(T["b"]) - n0(T["a"])
if INV_B is None or INV_A is None:
    print("Table storage: needs a before and an after table_inventory run (see section 1).")
else:
    print(f"Table storage on the root account: {human(T['b'])} -> {human(T['a'])}  saved {human(TBL_SAVED)} "
          f"(${usd(TBL_SAVED):,.0f}/month, ${usd(TBL_SAVED) * 12:,.0f}/year)")
    print(f"Files: {n0(T['fb']):,} -> {n0(T['fa']):,}  ({n0(T['fb']) - n0(T['fa']):,} removed); "
          f"tables dropped: {T['dropped']:,}")
    if T["live_measured"]:
        print(f"⚠️  {T['live_measured']:,} of {T['n']:,} tables have no on-disk size in one of the runs and are compared by "
              f"live size, which VACUUM does not change. Their VACUUM savings show in section 2 only.")

## 4 · File storage, before vs after (`dbfs_scan`)

Every directory the *before* scan sized, compared with the *after* scan:

- `both`: sized in both runs.
- `gone`: sized before and not in the after plan at all, so it was deleted (after = 0).
- `new`: appeared after the cleanup (growth, shown as a negative saving).
- `not sized`: one of the runs did not reach it. It is left out of every total.

Run both scans with the same `scan_depth` and `max_children`, or the leaf directories will not line up.

In [0]:
def scan_side(run, side):
    schema = f"path string, kind string, bytes_{side} long, files_{side} long"
    if run is None or scan_all is None:
        return spark.createDataFrame([], schema)
    return scan_all.filter((F.col("run_id") == run["run_id"]) & (F.col("storage_class") == "root")) \
        .dropDuplicates(["path"]).select("path", "kind", F.col("bytes").alias(f"bytes_{side}"),
                                         F.col("files").alias(f"files_{side}"))


def plan_paths(run, side):
    if run is None or plan_all is None:
        return spark.createDataFrame([], f"path string, in_plan_{side} boolean")
    return plan_all.filter(F.col("run_id") == run["run_id"]).select("path").distinct() \
        .withColumn(f"in_plan_{side}", F.lit(True))


# Table locations from the after inventory, or the before one: used to split dbfs:/ into table vs file storage
_inv_for_split = INV_A or INV_B
if _inv_for_split is not None:
    inv_all.filter((F.col("run_id") == _inv_for_split["run_id"]) & (F.col("storage_class") == "root")) \
        .select(F.regexp_replace(F.regexp_replace(F.when(F.col("location").startswith("abfss://"),
                                                         F.regexp_replace("location", r"^abfss://[^/]+", ""))
                                                  .otherwise(F.regexp_replace("location", r"^dbfs:", "")), r"/+$", ""),
                                 r"^$", "/").alias("loc")) \
        .distinct().createOrReplaceTempView("tbl_locs")
else:
    spark.createDataFrame([], "loc string").createOrReplaceTempView("tbl_locs")

sb, sa = scan_side(SCAN_B, "b"), scan_side(SCAN_A, "a")
FS = sb.join(sa, ["path", "kind"], "full_outer") \
    .join(plan_paths(SCAN_B, "b"), "path", "left").join(plan_paths(SCAN_A, "a"), "path", "left")
# A path sized before and missing from the after plan was deleted. A path planned but not sized on one side is
# unknown, and stays out of the totals rather than counting as a saving.
FS = FS.withColumn("status",
    F.when(F.col("bytes_b").isNotNull() & F.col("bytes_a").isNotNull(), "both")
     .when(F.col("bytes_b").isNotNull() & F.col("in_plan_a").isNull(), "gone")
     .when(F.col("bytes_a").isNotNull() & F.col("in_plan_b").isNull(), "new")
     .otherwise("not sized")) \
    .withColumn("bytes_a", F.when(F.col("status") == "gone", F.lit(0)).otherwise(F.col("bytes_a"))) \
    .withColumn("files_a", F.when(F.col("status") == "gone", F.lit(0)).otherwise(F.col("files_a")))
parts = F.split("path", "/")
FS = FS.withColumn("top", F.concat_ws("/", F.slice(parts, 1, 2))) \
       .withColumn("dir2", F.when(F.size(parts) > 2, F.concat_ws("/", F.slice(parts, 1, 3))).otherwise(F.col("path")))
locs = spark.table("tbl_locs").alias("l")
paths = FS.select("path").distinct().alias("p")
in_tbl = paths.join(F.broadcast(locs), (F.col("l.loc") == F.col("p.path"))
                    | F.col("l.loc").startswith(F.concat(F.col("p.path"), F.lit("/")))
                    | F.col("p.path").startswith(F.concat(F.col("l.loc"), F.lit("/"))), "inner") \
    .select(F.col("p.path").alias("path")).distinct().withColumn("in_tbl", F.lit(True))
FS = FS.join(in_tbl, "path", "left").fillna(False, ["in_tbl"])
FS.cache().createOrReplaceTempView("fs_cmp")

FS_TOP = spark.sql("""
    SELECT top AS directory,
           round(sum(CASE WHEN status != 'not sized' THEN coalesce(bytes_b, 0) END) / power(1024, 4), 3) AS before_tib,
           round(sum(CASE WHEN status != 'not sized' THEN coalesce(bytes_a, 0) END) / power(1024, 4), 3) AS after_tib,
           round(sum(CASE WHEN status != 'not sized' THEN coalesce(bytes_b, 0) - coalesce(bytes_a, 0) END)
                 / power(1024, 4), 3) AS saved_tib,
           usd(sum(CASE WHEN status != 'not sized' THEN coalesce(bytes_b, 0) - coalesce(bytes_a, 0) END))
                 AS saved_usd_per_month,
           sum(CASE WHEN status != 'not sized' THEN files_b END) AS files_before,
           sum(CASE WHEN status != 'not sized' THEN files_a END) AS files_after,
           count_if(status = 'gone') AS paths_gone, count_if(status = 'new') AS paths_new,
           count_if(status = 'not sized') AS paths_not_sized
    FROM fs_cmp GROUP BY top
    ORDER BY saved_tib DESC NULLS LAST
""").cache()
display(FS_TOP)

In [0]:
%sql
-- Second-level dbfs:/ directories that changed most. 'in_tables' marks folders holding registered tables.
SELECT dir2 AS directory,
       max(in_tbl) AS in_tables,
       round(sum(coalesce(bytes_b, 0)) / power(1024, 3), 2) AS before_gib,
       round(sum(coalesce(bytes_a, 0)) / power(1024, 3), 2) AS after_gib,
       round(sum(coalesce(bytes_b, 0) - coalesce(bytes_a, 0)) / power(1024, 3), 2) AS saved_gib,
       usd(sum(coalesce(bytes_b, 0) - coalesce(bytes_a, 0))) AS saved_usd_per_month,
       sum(files_b) AS files_before, sum(files_a) AS files_after,
       sum(coalesce(files_b, 0) - coalesce(files_a, 0)) AS files_removed,
       concat_ws(',', collect_set(status)) AS statuses
FROM fs_cmp c
WHERE status != 'not sized'
GROUP BY dir2
HAVING abs(sum(coalesce(bytes_b, 0) - coalesce(bytes_a, 0))) > 0
ORDER BY saved_gib DESC
LIMIT 200

In [0]:
F_ = spark.sql("""
    SELECT sum(coalesce(bytes_b, 0)) AS b, sum(coalesce(bytes_a, 0)) AS a,
           sum(coalesce(files_b, 0)) AS fb, sum(coalesce(files_a, 0)) AS fa,
           sum(CASE WHEN in_tbl THEN coalesce(bytes_b, 0) END) AS tb, sum(CASE WHEN in_tbl THEN coalesce(bytes_a, 0) END) AS ta,
           sum(CASE WHEN in_tbl THEN coalesce(files_b, 0) END) AS tfb, sum(CASE WHEN in_tbl THEN coalesce(files_a, 0) END) AS tfa
    FROM fs_cmp WHERE status != 'not sized'
""").first().asDict()
NOT_SIZED = spark.sql("SELECT count(*) n, sum(coalesce(bytes_b, bytes_a)) b FROM fs_cmp WHERE status = 'not sized'").first()
FS_TOTAL = dict(b=n0(F_["b"]), a=n0(F_["a"]), fb=n0(F_["fb"]), fa=n0(F_["fa"]))
FS_FILE = dict(b=FS_TOTAL["b"] - n0(F_["tb"]), a=FS_TOTAL["a"] - n0(F_["ta"]),
               fb=FS_TOTAL["fb"] - n0(F_["tfb"]), fa=FS_TOTAL["fa"] - n0(F_["tfa"]))
if SCAN_B is None or SCAN_A is None:
    print("File storage: needs a before and an after dbfs_scan run (see section 1).")
else:
    print(f"{'dbfs:/ (paths sized in both runs)':<40} {'before':>12} {'after':>12} {'saved':>12} {'$/month':>9} "
          f"{'files before':>14} {'files after':>14}")
    for label, d in (("All of dbfs:/", FS_TOTAL),
                     ("  of which file storage (not in tables)", FS_FILE),
                     ("  of which table folders", {k: FS_TOTAL[k] - FS_FILE[k] for k in FS_TOTAL})):
        print(f"{label:<40} {human(d['b']):>12} {human(d['a']):>12} {human(d['b'] - d['a']):>12} "
              f"{usd(d['b'] - d['a']):>9,.0f} {d['fb']:>14,} {d['fa']:>14,}")
    if NOT_SIZED["n"]:
        print(f"\n{NOT_SIZED['n']:,} paths were sized in only one of the runs ({human(NOT_SIZED['b'])}) and are left out. "
              f"Resume the incomplete 01a run to include them.")
    print("A directory is matched to a table when a table location is the directory, inside it, or above it, so a\n"
          "folder that only partly holds tables counts as table folder.")

## 5 · Account totals (`baseline`) and Azure

The `baseline` rows that `01_investigation` saved with the before and after inventories, and the Azure size.
Azure is the only figure that includes the containers notebooks cannot see, and it is what the bill uses.

In [0]:
base = table_or_none("baseline")


def base_row(run):
    if run is None or base is None:
        return None
    r = base.filter(F.col("run_id") == run["run_id"]).first()
    return r.asDict() if r else None


BASE_B, BASE_A = base_row(INV_B), base_row(INV_A)
ROWS = [("Visible through dbfs:/ (01a)", "scanned_root_bytes"),
        ("Registered tables on disk", "table_disk_bytes"),
        ("Live table data", "table_live_bytes"),
        ("VACUUM-able overhead (est.)", "vacuumable_bytes_est")]
AZ_B = AZURE_REPORTED_TIB * TIB
AZ_A = AZURE_NOW_TIB * TIB if AZURE_NOW_TIB is not None else None

In [0]:
print(f"{'':<34} {'before':>12} {'after':>12} {'change':>12} {'$/month':>9}")
for label, k in ROWS:
    b = BASE_B.get(k) if BASE_B else None
    a = BASE_A.get(k) if BASE_A else None
    d = (b - a) if b is not None and a is not None else None
    print(f"{label:<34} {human(b):>12} {human(a):>12} {human(d):>12} {usd(d) if d is not None else 0:>9,.0f}")
if BASE_B and BASE_A:
    fb = n0(BASE_B["scanned_root_bytes"]) - n0(BASE_B["table_disk_bytes"])
    fa = n0(BASE_A["scanned_root_bytes"]) - n0(BASE_A["table_disk_bytes"])
    print(f"{'File storage (dbfs:/ minus tables)':<34} {human(fb):>12} {human(fa):>12} {human(fb - fa):>12} {usd(fb - fa):>9,.0f}")
    print(f"\nScan coverage of those baselines: before {n0(BASE_B.get('scan_coverage')):.0%}, "
          f"after {n0(BASE_A.get('scan_coverage')):.0%}")

print(f"\n{'Azure account size':<34} {human(AZ_B):>12} {human(AZ_A):>12} "
      f"{human(AZ_B - AZ_A) if AZ_A is not None else '-':>12} {usd(AZ_B - AZ_A) if AZ_A is not None else 0:>9,.0f}")
if AZ_A is None:
    print("Enter today's size from the Azure portal (storage account → Metrics → Used capacity) in azure_now_tib "
          "to see the real saving. It also catches the containers notebooks cannot see.")
else:
    print(f"Azure says the bill drops from ${AZURE_MONTHLY_COST_USD:,.0f} to about ${usd(AZ_A):,.0f}/month "
          f"(${usd(AZ_B - AZ_A) * 12:,.0f}/year) at the May 2026 rate.")

## 6 · Summary for the ticket

Copy the printed Markdown into the SEACO-6161 comment.

In [0]:
def money(b):
    return f"${usd(b):,.0f}/month (${usd(b) * 12:,.0f}/year)"


md = [f"### SEACO-6161 savings · `{WORKSPACE_NAME}` (`{ROOT_STORAGE_ACCOUNT}`)", ""]
md.append(f"Effective rate ${RATE * GIB:.5f}/GiB-month (${AZURE_MONTHLY_COST_USD:,.2f} for {AZURE_REPORTED_TIB} TiB, May 2026).")
if FIRST_TS is not None:
    md.append(f"Cleanup ran from {FIRST_TS:%Y-%m-%d} to {LAST_TS:%Y-%m-%d}: {done.count():,} actions.")
md += ["", "**Removed by the cleanup (measured when each step ran)**", "",
       "| Storage | Freed | Files removed | Saving |", "|---|---:|---:|---|"]
for k, label in (("table", "Table storage"), ("file", "File storage (`dbfs:/` folders)")):
    b, f = n0(ACT[k].get("freed")), n0(ACT[k].get("files"))
    md.append(f"| {label} | {human(b)} | {f:,} | {money(b)} |")
md.append(f"| **Total** | **{human(ACT_FREED)}** | **{ACT_FILES:,}** | **{money(ACT_FREED)}** |")

md += ["", "| Step | Actions | Freed | Files removed | $/month |", "|---|---:|---:|---:|---:|"]
for r in STEP_SUMMARY.collect():
    md.append(f"| `{r.step}` | {r.actions:,} | {r.freed_tib or 0:,.3f} TiB | {r.files_removed or 0:,} | "
              f"${r.saved_usd_per_month or 0:,.0f} |")

if INV_B is not None and INV_A is not None:
    md += ["", f"**Table storage, before vs after** (inventory {INV_B['run_ts']:%Y-%m-%d} → {INV_A['run_ts']:%Y-%m-%d}): "
               f"{human(T['b'])} → {human(T['a'])}, saved {human(TBL_SAVED)} = {money(TBL_SAVED)}; "
               f"files {n0(T['fb']):,} → {n0(T['fa']):,}; {T['dropped']:,} tables dropped."]
if SCAN_B is not None and SCAN_A is not None:
    md += ["", f"**File storage, before vs after** (dbfs:/ scan {SCAN_B['run_ts']:%Y-%m-%d} → {SCAN_A['run_ts']:%Y-%m-%d}): "
               f"{human(FS_FILE['b'])} → {human(FS_FILE['a'])}, saved {human(FS_FILE['b'] - FS_FILE['a'])} = "
               f"{money(FS_FILE['b'] - FS_FILE['a'])}; files {FS_FILE['fb']:,} → {FS_FILE['fa']:,}."]
    md += ["", "| Top directories | Before TiB | After TiB | Saved TiB | $/month |", "|---|---:|---:|---:|---:|"]
    for r in FS_TOP.limit(10).collect():
        md.append(f"| `{r.directory}` | {r.before_tib or 0:,.3f} | {r.after_tib or 0:,.3f} | {r.saved_tib or 0:,.3f} | "
                  f"${r.saved_usd_per_month or 0:,.0f} |")
if AZ_A is not None:
    md += ["", f"**Azure account size**: {AZURE_REPORTED_TIB:,.2f} TiB → {AZURE_NOW_TIB:,.2f} TiB, "
               f"saving {money(AZ_B - AZ_A)}."]
else:
    md += ["", "_Azure account size after cleanup not entered yet; the figures above come from the notebooks._"]
REPORT_MD = "\n".join(md)

In [0]:
print(REPORT_MD)